# Tank 0.1 — demo do `tank check`

**A tese em uma frase:** o projeto declara a ontologia dele em código Python, e o Tank
verifica — deterministicamente, sem LLM — se o banco SurrealDB respeita o que foi declarado.
Se não respeita, o build quebra.

O que esta demo mostra:
1. Uma ontologia **mínima** e o check passando (golden)
2. As **sabotagens**: tabela errada, campo errado, vocabulário divergente, ontologia quebrada
3. A ontologia **completa** (um acervo de notícias: grafo + vetor + full-text + freshness)
4. Os estados que ninguém mais reporta: **VACUOUS** (tabela vazia nunca passa em silêncio)
5. Como isso roda no CI (exit codes, JSON)

> Referência dos códigos (`TBL-001`, `VEC-002`…): [`docs/checks.md`](../docs/checks.md)

In [1]:
# Setup: conexão com o SurrealDB v3 local e seeds da demo
import base64, json, urllib.request
from pathlib import Path

from tank import (
    Attr,
    FullText,
    Locator,
    Ontology,
    OntologyError,
    Relation,
    Scope,
    StableId,
    UnitType,
    Vector,
    Weight,
    Freshness,
)
from tank.checks import run_check

URL, USER, PASSWORD, NS = "http://127.0.0.1:8019", "root", "root", "tank_demo"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())


def sql(db, statements, allow_errors=False):
    auth = base64.b64encode(f"{USER}:{PASSWORD}".encode()).decode()
    req = urllib.request.Request(
        f"{URL}/sql",
        data=statements.encode(),
        headers={
            "Authorization": f"Basic {auth}",
            "Accept": "application/json",
            "surreal-ns": NS,
            "surreal-db": db,
        },
    )
    with urllib.request.urlopen(req, timeout=20) as r:
        results = json.load(r)
    errors = [x for x in results if x.get("status") != "OK"]
    assert allow_errors or not errors, errors[:2]
    return results


def fresh_db(db, fixture=None):
    sql(db, f"REMOVE DATABASE IF EXISTS `{db}`;", allow_errors=True)
    sql(
        db, "DEFINE PARAM $bootstrap VALUE 1;", allow_errors=True
    )  # ns/db só materializam com write
    sql(db, "DEFINE PARAM $bootstrap2 VALUE 1;")
    if fixture:
        sql(db, (ROOT / "tests" / "fixtures" / fixture / "seed.surql").read_text())


async def check(ontology, db):
    return await run_check(
        ontology, url=URL, namespace=NS, database=db, user=USER, password=PASSWORD
    )


def so_problemas(report):
    "Versão compacta: só o que não é PASS."
    for f in report.findings:
        if f.status != "PASS":
            print(f"  {f}")
    print(f"\n  exit code: {report.exit_code()}")


fresh_db("demo_minima", "degrau1")
fresh_db("demo_completa", "noticias_mini")
print("Seeds aplicados em", URL, "· ns", NS)

Seeds aplicados em http://127.0.0.1:8019 · ns tank_demo


## 1. A ontologia mínima

Um tipo do domínio (`laudo`), mapeado para a tabela onde ele mora (`parecer_tecnico` —
nomes diferentes **de propósito**: o agente tem que ler a declaração, não chutar).
Os `attrs` declaram os campos consultáveis: o que o agente pode usar num `WHERE`/`ORDER BY`,
com o vocabulário fechado de valores.

In [2]:
minima = Ontology(
    types=[
        UnitType(
            "laudo",
            table="parecer_tecnico",
            nature="original",
            id=StableId.of("codigo"),
            text="corpo_texto",
            locator=Locator(source="codigo"),
            attrs=[
                Attr("situacao", "string", values=["vigente", "revogado"]),
                Attr("emitido_em", "datetime"),
            ],
        ),
    ],
)
print(minima.to_json()[:600], "…")  # o export JSON: a superfície que a skill entrega ao agente

{
  "types": [
    {
      "name": "laudo",
      "table": "parecer_tecnico",
      "nature": "original",
      "id": {
        "fields": [
          "codigo"
        ]
      },
      "text": "corpo_texto",
      "attrs": [
        {
          "name": "situacao",
          "type": "string",
          "values": [
            "vigente",
            "revogado"
          ],
          "description": null
        },
        {
          "name": "emitido_em",
          "type": "datetime",
          "values": null,
          "description": null
        }
      ],
      "locator": {
        "source": "c …


## 2. O check golden — banco e ontologia coerentes

In [3]:
report = await check(minima, "demo_minima")
print(report.render_text())

tank check
  server:    http://127.0.0.1:8019 (surrealdb-3.1.6+20260813.cfbaec4)
  target:    ns=tank_demo db=demo_minima
  tables:    parecer_tecnico=12

  PASS    TBL-001  type:laudo: table 'parecer_tecnico' exists (SCHEMAFULL, kind=normal)
  PASS    FLD-001  type:laudo: field 'codigo' is defined on 'parecer_tecnico'
  PASS    FLD-001  type:laudo: field 'corpo_texto' is defined on 'parecer_tecnico'
  PASS    FLD-001  type:laudo: attr 'emitido_em': declared 'datetime' matches DEFINE FIELD type 'datetime'
  PASS    FLD-001  type:laudo: attr 'situacao': declared 'string' matches DEFINE FIELD type 'string'
  PASS    ATTR-010 type:laudo: attr 'situacao': sampled values within declared vocabulary

  6 pass, 0 fail, 0 warn, 0 vacuous

  A PASS here means: the declaration is not contradicted by the schema
  and the sampled data of THIS environment. Not verified:
    - semantics of names — whether a declared type/relation means what you think it means
    - embedding model identity — only the

## 3. Sabotagem: a tabela declarada não existe

O erro canônico: a ontologia diz *"pessoa torce para time"* — e a tabela `torce` não existe.
*"Você está me dando uma ontologia que não existe no banco."*

In [4]:
sabotada = minima.model_copy(deep=True)
sabotada.types[0].table = "parecer_tecnico_v2"  # <- não existe
so_problemas(await check(sabotada, "demo_minima"))

  FAIL    TBL-001  type:laudo: type 'laudo' declares table 'parecer_tecnico_v2', which does not exist in the database

  exit code: 1


## 4. Sabotagem: o campo de texto tem outro nome

In [5]:
sabotada = minima.model_copy(deep=True)
sabotada.types[0].text = "corpo"  # <- o campo real chama corpo_texto
so_problemas(await check(sabotada, "demo_minima"))

  FAIL    FLD-002  type:laudo: field 'corpo' has no DEFINE FIELD on 'parecer_tecnico' and is absent from every sampled row

  exit code: 1


## 5. Vocabulário divergente — WARN, não FAIL

A ontologia declara menos valores do que os dados têm. Não é contradição estrutural,
é suspeita: vira aviso (que o `--strict` promove a erro no CI).

In [6]:
estreita = Ontology(
    types=[
        UnitType(
            "laudo",
            table="parecer_tecnico",
            text="corpo_texto",
            attrs=[Attr("situacao", "string", values=["vigente"])],
        )
    ]
)  # faltou 'revogado'
so_problemas(await check(estreita, "demo_minima"))

  WARN    ATTR-010 type:laudo: attr 'situacao': sampled values ['revogado'] are outside the declared vocabulary ['vigente']

  exit code: 0


## 6. Ontologia internamente quebrada — o build quebra SEM banco

Referência cruzada que não resolve explode **no import**, com todas as violações de uma vez.
É o "se a ontologia quebrar, o CI quebra" — antes de existir conexão.

In [7]:
try:
    Ontology(
        types=[UnitType("laudo", table="parecer_tecnico")],
        relations=[Relation("emitido_por", "laudo", "responsavel")],  # tipo não declarado
        scopes=[Scope("obra", via="pertence_a")],  # relação não declarada
    )
except OntologyError as e:
    print(e)

ontology is invalid (2 violation(s)):
  [ONT-002] relation 'emitido_por': to='responsavel' is not a declared unit type (declared: ['laudo'])
  [ONT-003] scope 'obra': via='pertence_a' is not a declared relation


## 7. A ontologia completa — um acervo de notícias com grafo

4 tipos (`noticia`, `entidade`, `tema`, `documento`), 3 arestas com direção declarada,
peso na aresta, busca vetorial (dimensão + métrica), full-text (analyzer + língua),
escopo e freshness. **O índice não é declarado — é derivado**: declarar `Vector(dim=4)`
faz o check exigir o índice HNSW correspondente.

In [8]:
completa = Ontology(
    types=[
        UnitType(
            "noticia",
            table="noticia",
            nature="original",
            id=StableId.of("titulo"),
            text="corpo",
            attrs=[Attr("titulo", "string"), Attr("publicado_em", "datetime")],
            vector=Vector("emb", 4, "cosine"),
            fulltext=FullText("corpo", analyzer="az_pt", language="portuguese"),
        ),
        UnitType(
            "entidade",
            table="entidade",
            attrs=[
                Attr("nome", "string"),
                Attr("tipo", "string", values=["pessoa", "orgao", "organizacao"]),
            ],
        ),
        UnitType("tema", table="tema", attrs=[Attr("nome", "string")]),
        UnitType(
            "documento",
            table="documento",
            attrs=[
                Attr("numero", "string"),
                Attr("estado", "string", values=["tramitando", "aprovado", "rejeitado"]),
            ],
        ),
    ],
    relations=[
        Relation("fala_de", "noticia", "entidade", weight=Weight("peso")),
        Relation("sobre", "noticia", "tema"),
        Relation("cita", "noticia", "documento"),
    ],
    scopes=[Scope("entidade", via="fala_de")],
    freshness=[Freshness("noticia", "publicado_em", "30d")],
)
report = await check(completa, "demo_completa")
print(report.render_text())

tank check
  server:    http://127.0.0.1:8019 (surrealdb-3.1.6+20260813.cfbaec4)
  target:    ns=tank_demo db=demo_completa
  tables:    cita=2, documento=2, entidade=2, fala_de=3, noticia=3, sobre=2, tema=2

  PASS    TBL-001  type:noticia: table 'noticia' exists (SCHEMAFULL, kind=normal)
  PASS    FLD-001  type:noticia: field 'corpo' is defined on 'noticia'
  PASS    FLD-001  type:noticia: field 'emb' is defined on 'noticia'
  PASS    FLD-001  type:noticia: attr 'publicado_em': declared 'datetime' matches DEFINE FIELD type 'datetime'
  PASS    FLD-001  type:noticia: attr 'titulo': declared 'string' matches DEFINE FIELD type 'string'
  PASS    VEC-002  type:noticia: index 'idx_vec' DIMENSION matches (4)
  PASS    VEC-003  type:noticia: index 'idx_vec' DIST matches (cosine)
  PASS    VEC-010  type:noticia: sampled embedding dimensions all match 4
  PASS    FTS-001  type:noticia: FTS index 'idx_fts' covers 'corpo' (analyzer: az_pt)
  PASS    TBL-001  type:entidade: table 'entidade' exis

## 8. Sabotagens na completa

**8a. Dimensão declarada ≠ índice.** A ontologia diz 8, o índice tem 4 — e a amostra
dos embeddings também tem 4 (dimensão mista é um modo de falha real e silencioso de produção).

In [9]:
errada = completa.model_copy(deep=True)
errada.types[0].vector = Vector("emb", 8, "cosine")
so_problemas(await check(errada, "demo_completa"))

  FAIL    VEC-002  type:noticia: index 'idx_vec' has DIMENSION 4, ontology declares dim=8
  FAIL    VEC-010  type:noticia: sampled embeddings in 'emb' have dimensions [4], expected 8 — rows ingested before the index existed are not defended by the server

  exit code: 1


**8b. Índice vetorial removido do banco** — a busca declarada ficou sem motor.

In [10]:
fresh_db("demo_sab", "noticias_mini")
sql("demo_sab", "REMOVE INDEX idx_vec ON TABLE noticia;")
so_problemas(await check(completa, "demo_sab"))

  FAIL    VEC-001  type:noticia: vector declared on field 'emb' but no ANN index (HNSW/MTREE/DISKANN) covers it on 'noticia' — vector search would be a full scan or an error

  exit code: 1


**8c. Aresta removida** — a mesma classe de erro, agora sobre relação.

In [11]:
fresh_db("demo_sab", "noticias_mini")
sql("demo_sab", "REMOVE TABLE cita;")
so_problemas(await check(completa, "demo_sab"))

  FAIL    REL-001  relation:cita: relation 'cita' declares edge table 'cita', which does not exist in the database

  exit code: 1


**8d. Aresta criada "no improviso"** (só com `RELATE`, sem `DEFINE TABLE ... TYPE RELATION`).
Funciona, mas nada defende a direção no write — o check aprova por amostragem e **recomenda** explicitar.

In [12]:
fresh_db("demo_sab", "noticias_mini")
sql("demo_sab", "REMOVE TABLE sobre; RELATE noticia:n1->sobre->tema:t1;")
so_problemas(await check(completa, "demo_sab"))

  WARN    REL-002  relation:sobre: edge 'sobre' endpoints match by sampling, but the table is TYPE ANY — nothing defends direction on write; recommend DEFINE TABLE sobre TYPE RELATION IN noticia OUT tema

  exit code: 0


## 9. Tabela vazia — VACUOUS, nunca PASS silencioso

Schema perfeito com zero linhas passa em todo check estrutural "por vacuidade".
O Tank reporta isso como um estado próprio — no CI, `--strict` o trata como falha.

In [13]:
fresh_db("demo_vazia")
ddl = "\n".join(
    l
    for l in (ROOT / "tests/fixtures/degrau1/seed.surql").read_text().splitlines()
    if l.startswith("DEFINE")
)
sql("demo_vazia", ddl)
report = await check(minima, "demo_vazia")
so_problemas(report)
print(f"  exit code com --strict: {report.exit_code(strict=True)}")

  VACUOUS VAC-001  type:laudo: table 'parecer_tecnico' has 0 rows — every sampling check for this type is vacuous; structural checks still apply

  exit code: 0
  exit code com --strict: 1


## 10. No CI

```bash
uv run tank check --ontology ontology.py --url $SURREAL_URL --ns prod --db app --strict --json
```

Exit `0` = coerente · `1` = banco contradiz a ontologia · `2` = ontologia inválida ou conexão.
O `--json` entrega o relatório estruturado para máquinas:

In [14]:
report = await check(minima, "demo_minima")
print(json.dumps(json.loads(report.model_dump_json())["findings"][0], indent=2, ensure_ascii=False))

{
  "code": "TBL-001",
  "status": "PASS",
  "subject": "type:laudo",
  "table": "parecer_tecnico",
  "message": "table 'parecer_tecnico' exists (SCHEMAFULL, kind=normal)"
}


---

## O que vem depois (fora do 0.1)

- **0.2**: o decorator de acesso (`@access_tool`) — registro de uso e exceptions, a matéria-prima de eval/analytics
- **Teste de geração**: o agente lê ontologia + skill e gera a função de acesso — golden/sabotagem/**ablação** com LLM

Referência completa dos códigos: [`docs/checks.md`](../docs/checks.md)